In [ ]:
# local imports
from picell_1d import *
from dist_gen import *
# external imports
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

# add ffmpeg
mpl.rcParams['animation.ffmpeg_path'] = "C:\\Program Files\\FFmpeg\\bin\\ffmpeg.exe"

In [ ]:
save_figs = False

sweep_params: dict = {
	"x_min":			-0.00005,
	"x_max":			 0.00005,
	"v_min":			-10,
	"v_max":			 10,
	"steps_per_pd":		 75,
	"n_frames":			 500,
	"window_res":		 250,
	"f_cells_per_lam_D": 2000,
    "max_field_res":	 5000,
    "fixed_ions":		 True,
    "two_stream":		 False,
	"rho_lim_scale":	 15,
	"E_lim_scale":		 10,
	"ec_lim_margin":	 10,
	"N_e":				 50000,
	"N_i":				 50000,
	"x_pert_amp_e":		 0.1,
	"x_pert_amp_i":		 0,
    "x_pert_mode":		 1,
	"v_th_ratio_e":		 1/5000,
    "v_th_ratio_i":		 1/5000,
    "v_dr_ratio_e":		 0,
	"v_dr_ratio_i":		 0,
    "E_solver":			 1,
    "integration":		 0
}

pic_1d(sweep_params, r"outputs\langmuir.mp4", show_plot = True, verbose = True)

In [ ]:
# Fixed box length
L = 0.0001
# array of total particle counts
N_s_arr = np.logspace(2, 5, 100).astype(int)
# convert to number densities
n_s_arr = N_s_arr / L
# define number of samples taken at each density
pts_per_val = 1
N_samples = pts_per_val * len(N_s_arr)

sweep_data = {
	"densities": [],
	"numbers": [],
	"frequencies": []
}

# sweeping loop
for i in range(len(N_s_arr)):
	sweep_params["x_min"] = -L/2
	sweep_params["x_max"] = L/2
	sweep_params["N_e"] = N_s_arr[i]
	sweep_params["N_i"] = N_s_arr[i]
	for j in range(pts_per_val):
		sample_idx = i * pts_per_val + j + 1
		print(f"\nSample {sample_idx}/{N_samples}")
		result = pic_1d(sweep_params, show_plot = False, verbose = False)
		sweep_data["densities"].append(n_s_arr[i])
		sweep_data["numbers"].append(N_s_arr[i])
		sweep_data["frequencies"].append(result["w_obs"])

In [ ]:
def sqrt_fit(x, A):
	return A * np.sqrt(x)

plt.scatter(sweep_data["densities"], sweep_data["frequencies"], marker=".", label = "Samples")
if fixed_ions:
	plt.title("Observed Plasma Frequency over Number Density")
else:
	plt.title("Plasma Frequency over Number Density (Mobile Ions)")
plt.xlabel("Number Density $n_0$ (particles/m)")
plt.ylabel("Plasma Frequency $\omega_p$ (rad/s)")
popt, pcov = curve_fit(sqrt_fit, sweep_data["densities"], sweep_data["frequencies"], p0 = [1], maxfev = 10000)
n0_space = np.linspace(min(sweep_data["densities"]), max(sweep_data["densities"]), 500)
plt.plot(n0_space, sqrt_fit(n0_space, popt[0]), "--r", label = "Curve Fit")
plt.legend()

A = popt[0]
A_real = np.sqrt(e**2 / (m_e * e_0))
sigma_A = np.sqrt(np.diag(pcov)[0])
print(f"A value: {A} ± {sigma_A}")
print(f"Real Value: {A_real}")
print(100 * ((A/A_real)-1))
if fixed_ions and save_figs:
	plt.savefig(r"outputs\sweeps\n_0_sweep.png")
else:
	plt.savefig(r"outputs\sweeps\n_0_sweep_i.png")